URBAN SEGMENTATION

In [ ]:
from google.colab import drive
import os

# Montar el drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
from google.colab import drive
from osgeo import gdal
import numpy as np
import matplotlib.pyplot as plt

# 1. Rutas (Asegúrate de que 'ruta_terreno' y 'ruta_segmentacion' sigan siendo las correctas)
ruta_terreno = '/content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E2/Terrain Model/'
ruta_segmentacion = '/content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E2/Urban Segmentation/'

template_path = os.path.join(ruta_terreno, 'Mosaico_Maestro_CEM_1.5m.tif')
urbana_in = os.path.join(ruta_segmentacion, 'capa_urbana.tif')
infra_in = os.path.join(ruta_segmentacion, 'capa_infraestructura.tif')

# Salidas
mask_out = os.path.join(ruta_segmentacion, 'Mascara_Urbana_Refinamiento.tif')
vias_out = os.path.join(ruta_segmentacion, 'Vialidades_Principales_Segmentadas.tif')
png_out = os.path.join(ruta_segmentacion, 'Manzanas_Morfologia_Detalle.png')

def align_with_gdal(input_file, output_file, template_file):
    # Obtenemos la información de la plantilla (Master Grid)
    temp_ds = gdal.Open(template_file)
    projection = temp_ds.GetProjection()
    geotransform = temp_ds.GetGeoTransform()
    width = temp_ds.RasterXSize
    height = temp_ds.RasterYSize

    # Configuramos la reproyección/alineación
    options = gdal.WarpOptions(
        format='GTiff',
        outputBounds=[geotransform[0], geotransform[3] + geotransform[5]*height,
                      geotransform[0] + geotransform[1]*width, geotransform[3]],
        xRes=geotransform[1],
        yRes=geotransform[5],
        dstSRS=projection,
        resampleAlg=gdal.GRA_NearestNeighbour, # Mantiene integridad de clases
        dstNodata=-9999,
        creationOptions=['COMPRESS=DEFLATE', 'TILED=YES'] # Optimiza lectura posterior
    )

    print(f"Sincronizando: {os.path.basename(input_file)}...")
    gdal.Warp(output_file, input_file, options=options)

# Ejecución
try:
    align_with_gdal(urbana_in, mask_out, template_path)
    align_with_gdal(infra_in, vias_out, template_path)
    print("✅ Sincronización exitosa.")

    # Generar visualización rápida
    ds = gdal.Open(mask_out)
    data = ds.ReadAsArray()
    data = np.where(data > 0, 1, 0)

    plt.figure(figsize=(10, 10))
    plt.imshow(data, cmap='Blues')
    plt.title('Segmentación de Morfología Urbana - Veracruz')
    plt.axis('off')
    plt.savefig(png_out, dpi=300)
    plt.show()

except Exception as e:
    print(f"❌ Error persistente: {e}")
    print("Sugerencia: Si el error persiste, intenta descargar el archivo 'capa_urbana.tif' y volverlo a subir a Drive; podría estar corrupto.")

/usr/local/lib/python3.12/dist-packages/osgeo/gdal.py:312: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


Sincronizando: capa_urbana.tif...
Sincronizando: capa_infraestructura.tif...
✅ Sincronización exitosa.


MORFOLOGIA URBANA

In [ ]:
import os
from osgeo import gdal
import matplotlib.pyplot as plt
import numpy as np

# 1. Definir rutas de carpetas y archivos
ruta_segmentacion = '/content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E2/Urban Segmentation/'
input_mask = os.path.join(ruta_segmentacion, 'Mascara_Urbana_Refinamiento.tif')

# Salidas
output_tif_final = os.path.join(ruta_segmentacion, 'Capa_Manzanas_Segmentadas.tif')
output_png_final = os.path.join(ruta_segmentacion, 'Manzanas_Morfologia_Detalle.png')

print("Abriendo máscara para generación de entregables...")

# 2. Generar el .TIF (Copia técnica optimizada para E3/PyTorch)
# Usamos gdal.Translate para asegurar que el archivo tenga compresión y sea ligero
gdal.Translate(output_tif_final, input_mask,
               creationOptions=['COMPRESS=DEFLATE', 'TILED=YES'])
print(f"✅ Archivo técnico generado: {output_tif_final}")

# 3. Generar el .PNG (Visualización académica para el reporte)
ds = gdal.Open(input_mask)
data = ds.ReadAsArray()

# Normalización para visualización: 1 para urbano, 0 para fondo
# Ignoramos el valor NoData (-9999) para que no oscurezca la imagen
viz_data = np.where((data > 0) & (data != -9999), 1, 0)

plt.figure(figsize=(12, 10))
# Usamos el mapa de color 'viridis' o 'Blues' para una apariencia profesional
plt.imshow(viz_data, cmap='Blues', interpolation='nearest')

# Formato de tesis
plt.title('E2: Detalle de Morfología Urbana\nPuerto de Veracruz, Veracruz', fontsize=14, pad=15)
plt.axis('off') # Eliminar ejes para un acabado de mapa limpio

# Guardar en alta resolución (300 DPI) para evitar pixelado en el documento final
plt.savefig(output_png_final, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print(f"✅ Visualización generada: {output_png_final}")

Abriendo máscara para generación de entregables...


/usr/local/lib/python3.12/dist-packages/osgeo/gdal.py:312: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


✅ Archivo técnico generado: /content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E2/Urban Segmentation/Capa_Manzanas_Segmentadas.tif


In [ ]:
import os
from osgeo import gdal
import matplotlib.pyplot as plt
import numpy as np

# 1. Ruta del archivo procesado en la etapa anterior
ruta_segmentacion = '/content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E2/Urban Segmentation/'
input_mask = os.path.join(ruta_segmentacion, 'Mascara_Urbana_Refinamiento.tif')
output_png = os.path.join(ruta_segmentacion, 'Manzanas_Morfologia_Detalle.png')

print(f"Leyendo datos de: {input_mask}")

# 2. Abrir el dataset con GDAL
ds = gdal.Open(input_mask)
band = ds.GetRasterBand(1)
data = band.ReadAsArray()

# 3. Limpieza y preparación para visualización
# Convertimos a binario (1 para urbano, 0 para fondo) y manejamos el NoData
data = np.where((data > 0) & (data != -9999), 1, 0)

# 4. Configuración estética de la imagen
plt.figure(figsize=(15, 12)) # Tamaño grande para capturar detalle de manzanas
plt.imshow(data, cmap='viridis', interpolation='nearest') # 'viridis' da un estilo de mapa de calor profesional

# Añadir elementos de formato académico
plt.title('Morfología Urbana y Delimitación de Manzanas\nPuerto de Veracruz', fontsize=16, pad=20)
plt.colorbar(label='Presencia Urbana (Binaria)', ticks=[0, 1])
plt.axis('off') # Quitamos los ejes de píxeles para que parezca un mapa limpio

# 5. Guardar en alta resolución (300 DPI)
print(f"Generando exportación visual en: {output_png}")
plt.savefig(output_png, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("✅ Archivo PNG generado exitosamente.")

Leyendo datos de: /content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E2/Urban Segmentation/Mascara_Urbana_Refinamiento.tif


/usr/local/lib/python3.12/dist-packages/osgeo/gdal.py:312: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


Generando exportación visual en: /content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E2/Urban Segmentation/Manzanas_Morfologia_Detalle.png


In [ ]:
import os
from osgeo import gdal
import matplotlib.pyplot as plt
import numpy as np

# 1. Definir rutas
ruta_segmentacion = '/content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E2/Urban Segmentation/'
input_tif_path = os.path.join(ruta_segmentacion, 'Capa_Manzanas_Segmentadas.tif')
output_png_path = os.path.join(ruta_segmentacion, 'Capa_Manzanas_Segmentadas_Visualizacion.png')

print(f"Leyendo datos de: {input_tif_path}")

# 2. Abrir el dataset con GDAL
ds = gdal.Open(input_tif_path)
if ds is None:
    raise FileNotFoundError(f"No se pudo abrir el archivo: {input_tif_path}")

band = ds.GetRasterBand(1)
data = band.ReadAsArray()

# 3. Limpieza y preparación para visualización
# Asumiendo que los valores > 0 representan área urbana y -9999 es NoData
viz_data = np.where((data > 0) & (data != -9999), 1, 0)

# 4. Configuración estética de la imagen
plt.figure(figsize=(15, 12)) # Tamaño grande para una buena visualización
plt.imshow(viz_data, cmap='Blues', interpolation='nearest') # 'Blues' es un buen colormap para binario

plt.title('Capa de Manzanas Segmentadas (Visualización)\nPuerto de Veracruz', fontsize=16, pad=20)
plt.axis('off') # Eliminar ejes para una vista de mapa limpia

# 5. Guardar en alta resolución
print(f"Generando exportación visual en: {output_png_path}")
plt.savefig(output_png_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("✅ Archivo PNG generado exitosamente a partir de Capa_Manzanas_Segmentadas.tif.")

Leyendo datos de: /content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E2/Urban Segmentation/Capa_Manzanas_Segmentadas.tif


/usr/local/lib/python3.12/dist-packages/osgeo/gdal.py:312: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


Generando exportación visual en: /content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E2/Urban Segmentation/Capa_Manzanas_Segmentadas_Visualizacion.png
